# ZeroPoint against open decision models: Strands Decider and Clef-flash

Runs AWS's Strands Decider 2B and Cloudflare's Clef-flash (9B) on a GPU, on the same cases as the
LLM comparison: the 300-case sample of the three tests the models are unlikely to have seen (the 2,000-case test
and the two support-ticket tests), then the same cases with the options in another order. Each model is asked as
Laya is: the same instructions, with "None of these" added as one more option. The code is `build_strands` and
`build_clef_flash` in `benchmarks/compare/run.py`; this notebook only runs it.

Runtime: **GPU, A100** (Runtime → Change runtime type); both models also fit on an L4. No API key is needed: the
weights are public on Hugging Face.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
BRANCH = "v1-strands"      # "main" once the pull request is merged
%cd /content
!rm -rf zeropoint && git clone -q -b $BRANCH https://github.com/Javihaus/zeropoint.git
%cd /content/zeropoint
!pip install -q -e . datasets pandas scikit-learn requests
!pip install -q "strands-decider[cuda]==0.2.0" || pip install -q "strands-decider==0.2.0"

## 1. Rebuild the tests

The test texts are not in the repository: this rebuilds them from pinned dataset versions and checks their hashes.

In [ ]:
!cd /content/zeropoint && python benchmarks/compare/build_tests.py
!cd /content/zeropoint/benchmarks/compare/tests && sha256sum -c SHA256SUMS

## 2. Strands Decider 2B

About 5 GB of weights. 1,800 calls (900 cases, twice). Each model's weights are deleted from the disk after its run.

In [ ]:
import os, shutil, subprocess
os.chdir("/content/zeropoint")
TESTS = ["bench2000", "tickets_made", "tickets_far"]
HUB = os.path.expanduser("~/.cache/huggingface/hub")

def run(system, drop=()):
    env = dict(os.environ, DECIDER_DEVICE="cuda")
    p = subprocess.Popen(["python", "benchmarks/compare/run.py", system, *TESTS], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:                           # progress and errors appear in the notebook
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"{system} failed: the error is just above")
    for d in drop:                                  # free the disk for the next model
        shutil.rmtree(os.path.join(HUB, d), ignore_errors=True)

run("strands_none", drop=["models--StrandsAgents--strands-decider-2B-qwen3.5-v1-2610", "models--Qwen--Qwen3.5-2B-Base"])

## 3. Clef-flash (9B)

About 19 GB of weights.

In [ ]:
run("clef_flash_none", drop=["models--Cloudflare--clef-flash"])

## 4. Results

The same `report.py` as every other system: the section *Compared with LLMs and decision models*. Then download
the per-case outputs, to add them to the repository.

In [ ]:
!cd /content/zeropoint && python benchmarks/compare/report.py > /dev/null
text = open("/content/zeropoint/benchmarks/compare/results/tables.md").read()
print(text[text.index("## Compared with"):text.index("## The tests")])

In [ ]:
!cd /content/zeropoint/benchmarks/compare/out && zip -qr /content/deciders_results.zip strands_none clef_flash_none 2>/dev/null; ls -la /content/deciders_results.zip
try:
    from google.colab import files
    files.download("/content/deciders_results.zip")
except ImportError:
    pass